# Week 3 Exercise - Lecture to Study Pack

Upload audio or provide a YouTube link, transcribe with Whisper, then generate a study pack (summary, key concepts, quiz, flashcards).

In [ ]:
# optional installs
# !uv pip install -q openai-whisper gradio python-dotenv yt-dlp openai

In [5]:
import os
import re
import json
import sqlite3
import tempfile
import subprocess
from datetime import datetime

import gradio as gr
import whisper
from dotenv import load_dotenv
from openai import OpenAI

In [7]:
load_dotenv(override=True)

MODEL_OPENROUTER = "openai/gpt-4o-mini"
DB_PATH = "week3_study_pack.db"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

_whisper_model = None

def get_whisper_model():
    global _whisper_model
    if _whisper_model is None:
        _whisper_model = whisper.load_model("base")
    return _whisper_model

In [8]:
def init_db():
    with sqlite3.connect(DB_PATH) as conn:
        cur = conn.cursor()
        cur.execute("""
            CREATE TABLE IF NOT EXISTS sessions (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                created_at TEXT NOT NULL,
                source_type TEXT NOT NULL,
                source_value TEXT NOT NULL,
                title TEXT,
                transcript TEXT NOT NULL,
                summary TEXT NOT NULL,
                key_concepts_json TEXT NOT NULL,
                quiz_json TEXT NOT NULL,
                flashcards_json TEXT NOT NULL
            )
        """)
        conn.commit()

def save_session(source_type, source_value, title, transcript, pack):
    with sqlite3.connect(DB_PATH) as conn:
        cur = conn.cursor()
        cur.execute("""
            INSERT INTO sessions (
                created_at, source_type, source_value, title, transcript, summary,
                key_concepts_json, quiz_json, flashcards_json
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            datetime.utcnow().isoformat(),
            source_type,
            source_value,
            title,
            transcript,
            pack["summary"],
            json.dumps(pack["key_concepts"], ensure_ascii=False),
            json.dumps(pack["quiz"], ensure_ascii=False),
            json.dumps(pack["flashcards"], ensure_ascii=False)
        ))
        conn.commit()

init_db()

In [9]:
def run_cmd(cmd):
    subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

def download_youtube_audio(url):
    workdir = tempfile.mkdtemp(prefix="w3_youtube_")
    out_template = os.path.join(workdir, "audio.%(ext)s")
    cmd = [
        "yt-dlp",
        "-x",
        "--audio-format",
        "mp3",
        "--no-playlist",
        "-o",
        out_template,
        url
    ]
    run_cmd(cmd)
    audio_path = os.path.join(workdir, "audio.mp3")
    if not os.path.exists(audio_path):
        candidates = [f for f in os.listdir(workdir) if f.startswith("audio.")]
        if not candidates:
            raise RuntimeError("Failed to download audio from YouTube URL.")
        audio_path = os.path.join(workdir, candidates[0])
    return audio_path

def transcribe_audio(audio_path, language_name):
    language_map = {
        "Auto-detect": None,
        "English": "en",
        "Spanish": "es",
        "French": "fr",
        "German": "de",
        "Portuguese": "pt",
        "Arabic": "ar",
        "Swahili": "sw"
    }
    lang_code = language_map.get(language_name, None)
    model = get_whisper_model()
    result = model.transcribe(audio_path, language=lang_code, task="transcribe", verbose=False)
    return result.get("text", "").strip()

In [10]:
def parse_json_block(text):
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    match = re.search(r"\{[\s\S]*\}", text)
    if not match:
        raise ValueError("Model did not return JSON.")
    return json.loads(match.group(0))

def generate_study_pack(transcript, title, model_name):
    if not os.getenv("OPENROUTER_API_KEY"):
        raise RuntimeError("OPENROUTER_API_KEY is missing in .env")

    system = (
        "You create study packs from lecture transcripts. Return strict JSON only with keys: "
        "summary (string), key_concepts (array of strings), quiz (array of objects with question and answer), "
        "flashcards (array of objects with front and back)."
    )
    user = f"Title: {title or 'Untitled Lecture'}\n\nTranscript:\n{transcript}"

    response = client.chat.completions.create(
        model=model_name,
        temperature=0.2,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
    )

    raw = response.choices[0].message.content
    pack = parse_json_block(raw)

    pack.setdefault("summary", "")
    pack.setdefault("key_concepts", [])
    pack.setdefault("quiz", [])
    pack.setdefault("flashcards", [])
    return pack

def render_quiz_md(quiz):
    lines = []
    for i, item in enumerate(quiz, start=1):
        q = item.get("question", "")
        a = item.get("answer", "")
        lines.append(f"{i}. **Q:** {q}\n   **A:** {a}")
    return "\n\n".join(lines) if lines else "No quiz generated."

def render_flashcards_md(cards):
    lines = []
    for i, item in enumerate(cards, start=1):
        f = item.get("front", "")
        b = item.get("back", "")
        lines.append(f"{i}. **Front:** {f}\n   **Back:** {b}")
    return "\n\n".join(lines) if lines else "No flashcards generated."

def build_markdown(title, transcript, pack):
    concepts = "\n".join([f"- {c}" for c in pack.get("key_concepts", [])])
    quiz_md = render_quiz_md(pack.get("quiz", []))
    flash_md = render_flashcards_md(pack.get("flashcards", []))
    return f"# Study Pack: {title or 'Untitled Lecture'}\n\n## Summary\n{pack.get('summary','')}\n\n## Key Concepts\n{concepts}\n\n## Quiz\n{quiz_md}\n\n## Flashcards\n{flash_md}\n\n## Transcript\n{transcript}\n"

In [11]:
def create_study_pack(audio_file, youtube_url, title, language, model_name):
    try:
        if not audio_file and not (youtube_url or "").strip():
            return "", "", "", "", None, None, "Please upload audio or provide a YouTube URL."

        if audio_file:
            audio_path = audio_file
            source_type = "audio_upload"
            source_value = audio_file
        else:
            audio_path = download_youtube_audio(youtube_url.strip())
            source_type = "youtube"
            source_value = youtube_url.strip()

        transcript = transcribe_audio(audio_path, language)
        if not transcript:
            return "", "", "", "", None, None, "Transcription failed or was empty."

        pack = generate_study_pack(transcript, title, model_name)

        summary_md = pack.get("summary", "")
        quiz_md = render_quiz_md(pack.get("quiz", []))
        flashcards_md = render_flashcards_md(pack.get("flashcards", []))

        save_session(source_type, source_value, title, transcript, pack)

        out_dir = tempfile.mkdtemp(prefix="w3_pack_")
        md_path = os.path.join(out_dir, "study_pack.md")
        json_path = os.path.join(out_dir, "study_pack.json")

        with open(md_path, "w", encoding="utf-8") as f:
            f.write(build_markdown(title, transcript, pack))

        with open(json_path, "w", encoding="utf-8") as f:
            json.dump({
                "title": title,
                "transcript": transcript,
                "study_pack": pack
            }, f, indent=2, ensure_ascii=False)

        return transcript, summary_md, quiz_md, flashcards_md, md_path, json_path, "Done. Study pack generated and saved to SQLite."

    except Exception as e:
        return "", "", "", "", None, None, f"Error: {str(e)}"

In [ ]:
app = gr.Interface(
    fn=create_study_pack,
    inputs=[
        gr.Audio(type="filepath", label="Upload lecture audio (optional if using YouTube)"),
        gr.Textbox(label="YouTube URL (optional if uploading audio)", placeholder="https://www.youtube.com/watch?v=..."),
        gr.Textbox(label="Lecture title", placeholder="e.g. Intro to RAG"),
        gr.Dropdown(choices=["Auto-detect", "English", "Spanish", "French", "German", "Portuguese", "Arabic", "Swahili"], value="Auto-detect", label="Transcription language"),
        gr.Dropdown(choices=["openai/gpt-4o-mini", "meta-llama/llama-3.3-70b-instruct"], value="openai/gpt-4o-mini", label="Study-pack model (OpenRouter)")
    ],
    outputs=[
        gr.Textbox(label="Transcript", lines=12),
        gr.Markdown(label="Summary"),
        gr.Markdown(label="Quiz"),
        gr.Markdown(label="Flashcards"),
        gr.File(label="Download Markdown"),
        gr.File(label="Download JSON"),
        gr.Textbox(label="Status")
    ],
    title="Lecture to Study Pack Generator",
    description="Create study packs from uploaded audio or YouTube links.",
    flagging_mode="never"
)

app.launch()